In [7]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


In [8]:
import idna #implementacion de tecnicas de codificacion de strings (codificacion y decodificacion de caracterers unicode)

# def utf8_to_punycode(text: str) -> str:
#     """Encodes a UTF-8 string to its Punycode representation."""
#     return idna.encode(text).decode('ascii')

def punyencode(text: str) -> str:
    """Encodes a UTF-8 string to its Punycode representation, handling spaces by encoding each word separately."""
    
    return " ".join([idna.encode(word).decode('ascii') for word in text.split()])
    
def punydecode(punycode: str) -> str:
    """Decodes a Punycode string back to UTF-8."""
    #return idna.decode(punycode)
    return " ".join([idna.decode(word) for word in punycode.split()])

def process_name(name):
    name = name.lower()
    for n in name.split():
        if len(n) < 2:
            return ''
    try:
        return punyencode(name)
    except:
        #print(f'Cant convert {name}')
        return ''

dataset = open("city_names_full.txt", 'r', encoding='utf-8').read().split('\n')
with open('city_names_puny.txt', 'w', encoding='utf-8') as f:
    for n in dataset:
        name = process_name(n)
        if name != '':
            f.write(name + '\n')
dataset = open('city_names_puny.txt', 'r', encoding='utf-8').read().split('\n')
puny = [x for x in dataset if 'xn--' in x] #filtrar todos los que tienen caracteres raros
nopuny = [x for x in dataset if 'xn--' not in x]
np.random.seed(42)
dataset = [x.item() for x in np.random.choice(nopuny, 100000, replace=False)]

In [9]:
#construyendo dataset para makemore con MLPS

In [10]:
print(len(dataset))
charset = ['*'] + sorted(list(set([y for x in dataset for y in x])))
ctoi = {c:i for i, c in enumerate(charset)}
itoc = {i:c for i, c in enumerate(charset)}
charset_size = len(charset)


100000


In [11]:
context_size = 3
X, Y = [], []

for d in dataset[:1]:
    print(d)
    example = list(d) + ['*']
    context = [0] * context_size
    for c in example:
        print(''.join([itoc[x] for x in context])+ '---> + c')
        X.append(context)
        Y.append(ctoi[c])
        context = context[1:] + [ctoi[c]]

kotamangalam
***---> + c
**k---> + c
*ko---> + c
kot---> + c
ota---> + c
tam---> + c
ama---> + c
man---> + c
ang---> + c
nga---> + c
gal---> + c
ala---> + c
lam---> + c


In [12]:
X, Y

([[0, 0, 0],
  [0, 0, 24],
  [0, 24, 28],
  [24, 28, 33],
  [28, 33, 14],
  [33, 14, 26],
  [14, 26, 14],
  [26, 14, 27],
  [14, 27, 20],
  [27, 20, 14],
  [20, 14, 25],
  [14, 25, 14],
  [25, 14, 26]],
 [24, 28, 33, 14, 26, 14, 27, 20, 14, 25, 14, 26, 0])

In [13]:
#Red neuronal

In [14]:
g = torch.Generator().manual_seed(42)
emb_size = 2
C = torch.randn(charset_size, emb_size, generator=g)#genera una matriz de 40*2
C

tensor([[ 1.9269,  1.4873],
        [ 0.9007, -2.1055],
        [ 0.6784, -1.2345],
        [-0.0431, -1.6047],
        [-0.7521,  1.6487],
        [-0.3925, -1.4036],
        [-0.7279, -0.5594],
        [-0.7688,  0.7624],
        [ 1.6423, -0.1596],
        [-0.4974,  0.4396],
        [-0.7581,  1.0783],
        [ 0.8008,  1.6806],
        [ 1.2791,  1.2964],
        [ 0.6105,  1.3347],
        [-0.2316,  0.0418],
        [-0.2516,  0.8599],
        [-1.3847, -0.8712],
        [-0.2234,  1.7174],
        [ 0.3189, -0.4245],
        [ 0.3057, -0.7746],
        [-1.5576,  0.9956],
        [-0.8798, -0.6011],
        [-1.2742,  2.1228],
        [-1.2347, -0.4879],
        [-0.9138, -0.6581],
        [ 0.0780,  0.5258],
        [-0.4880,  1.1914],
        [-0.8140, -0.7360],
        [-1.4032,  0.0360],
        [-0.0635,  0.6756],
        [-0.0978,  1.8446],
        [-1.1845,  1.3835],
        [ 1.4451,  0.8564],
        [ 2.2181,  0.5232],
        [ 0.3466, -0.1973],
        [-1.0546,  1

In [15]:
#onehot es un vector que dependiendo de el numero que se ingrese va a mostrar 1 en el indice del caracter que se esta codeando (?)
#F.one_hot(torch.tensor(0), num_classes = charset_size).float()
F.one_hot(torch.tensor(0), num_classes = charset_size).float() @ C

tensor([1.9269, 1.4873])

In [16]:
F.one_hot(torch.tensor(X), num_classes = charset_size).float() @ C #devuelve una matriz con el embedding de cada uno de los char de X(pesos)

tensor([[[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873]],

        [[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [-0.9138, -0.6581]],

        [[ 1.9269,  1.4873],
         [-0.9138, -0.6581],
         [-1.4032,  0.0360]],

        [[-0.9138, -0.6581],
         [-1.4032,  0.0360],
         [ 2.2181,  0.5232]],

        [[-1.4032,  0.0360],
         [ 2.2181,  0.5232],
         [-0.2316,  0.0418]],

        [[ 2.2181,  0.5232],
         [-0.2316,  0.0418],
         [-0.4880,  1.1914]],

        [[-0.2316,  0.0418],
         [-0.4880,  1.1914],
         [-0.2316,  0.0418]],

        [[-0.4880,  1.1914],
         [-0.2316,  0.0418],
         [-0.8140, -0.7360]],

        [[-0.2316,  0.0418],
         [-0.8140, -0.7360],
         [-1.5576,  0.9956]],

        [[-0.8140, -0.7360],
         [-1.5576,  0.9956],
         [-0.2316,  0.0418]],

        [[-1.5576,  0.9956],
         [-0.2316,  0.0418],
         [ 0.0780,  0.5258]],

        [[-0.2316,  0

In [17]:
# (13, 3 ,40) @ (40, 2) --> (13, 3, 2)

In [18]:
C[[X]] #esto es el equivalente a la multiplicacion de matrices que devuelve la matriz de embeddings arriba

# Importante!
# • El tamaño contexto de nuestro modelo (cuántos caracteres se utilizan para predecir la distribución de probabilidad del siguiente) determina muchas cosas y es conveniente
# tener claro cual es este número. En nuestro ejemplo, el tamaño del contexto es 3. Es decir, utilizamos 3 caracteres para predecir el siguiente.
# • La matriz C puede interpretarse como la primer capa de perceptronoes.
# • C tiene 2 perceptrones de 40 entradas, es decir tiene 40 entradas y 2 salidas, lo que permite codificar 1 caracter en 2 dimensiones
# • Nuestro modelo no recibe como entrada un one hot de 40 dimensiones, recibe el índice de 3 caracteres.
# • Codificar 3 caracteres requiere que multiplicar (3, 40)@(40, 2) (3, 2) es decir, primero tenemos que calcular el one hot de los 3 caracteres y despues
# multiplicar los one hot por C.
# • Durante el entrenamiento vamos a querer mandarle "batches" de 3 caracteres. Es decir, vamos a queresr enviar varios grupos de 3 caracteres. Si el tamaño del grupo es
# de 13, como vimos en el ejemplo, podemos interpretar que capa C esta recibiendo 13 grupos de 3 caracteres encodeados en one hot cuya dimensión seria (13,
# 3, 40) y lo va a multiplicar por (40, 2), es decir (13, 3, 40)@(40, 2) (13, 3, 2). La salida se puede interpretar como 13 grupos de 3 caracteres encodeados en 2
# dimensiones (las dimensiones del embedding).
# • Se puede lograr el mismo efecto indexando C mediante el uso las ténicas avanzadas de indexing de PyTorch, en lugar de hacer operaciones con matrices (guiño, guiño).
# • En general las dimensiones del modelo van a ser de
# o Entrada: (batch_size, context_size, (charsetLSize)J0, charset_size será necesario si usamos one hot encondig, se puede obviar si usamos indexing.
# o Primer capa (Matriz de embeddings): (charset_size, es decir, una red de emb_size perceptrones (emb_size salidas), cada uno con charset_size
# entradas.
# o Salida: (batch_size, context_size,emb_size)
# o (batch_size, context_size, (charset_size)) @ (charset_size, emb_size) (batch_size, context_size,emb_size)
# • Es necesario tener presente el tamaño de la salida por que de eso depende el tamaño de la entrada de la siquiente capa.

C:\Users\TheCague\AppData\Local\Temp\ipykernel_10284\934146368.py:1: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:353.)
  C[[X]] #esto es el equivalente a la multiplicacion de matrices que devuelve la matriz de embeddings arriba


tensor([[[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873]],

        [[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [-0.9138, -0.6581]],

        [[ 1.9269,  1.4873],
         [-0.9138, -0.6581],
         [-1.4032,  0.0360]],

        [[-0.9138, -0.6581],
         [-1.4032,  0.0360],
         [ 2.2181,  0.5232]],

        [[-1.4032,  0.0360],
         [ 2.2181,  0.5232],
         [-0.2316,  0.0418]],

        [[ 2.2181,  0.5232],
         [-0.2316,  0.0418],
         [-0.4880,  1.1914]],

        [[-0.2316,  0.0418],
         [-0.4880,  1.1914],
         [-0.2316,  0.0418]],

        [[-0.4880,  1.1914],
         [-0.2316,  0.0418],
         [-0.8140, -0.7360]],

        [[-0.2316,  0.0418],
         [-0.8140, -0.7360],
         [-1.5576,  0.9956]],

        [[-0.8140, -0.7360],
         [-1.5576,  0.9956],
         [-0.2316,  0.0418]],

        [[-1.5576,  0.9956],
         [-0.2316,  0.0418],
         [ 0.0780,  0.5258]],

        [[-0.2316,  0

In [19]:
#Capa oculta (capa de perceptrones con una funcion de activacion) (capa de neuronas)
emb = C[[X]] #embeddings del dataset (pesos)

C:\Users\TheCague\AppData\Local\Temp\ipykernel_10284\501557912.py:2: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:353.)
  emb = C[[X]] #embeddings del dataset (pesos)


In [20]:

#definicion de la capa oculta
n_hidden = 128 #128 neuronas, cada una con sus entradas y salidas
W1 = torch.randn(emb_size * context_size,  n_hidden) #el contexto esta enbebido en 2 dimensiones y son 3 caracteres en el contexto, cantidad de perceptrones
b1 = torch.randn(n_hidden) #bias

In [21]:

emb [0, 0], emb[0,1], emb[0,2]
#forward pass
F.tanh(emb[0].view(-1, 6) @ W1 + b1 )#salida de la capa C por la capa de perceptrones, aplicandole la funcion de activacion(tanh)


tensor([[-0.9691,  0.9185,  1.0000, -0.9657, -0.9769, -0.2690, -0.9907, -0.9802,
          0.9612,  1.0000,  0.7341,  0.9988,  1.0000, -0.9385, -0.7218, -0.9999,
          0.9076, -0.6039, -0.9999, -0.9100, -1.0000,  1.0000,  0.9886, -0.9999,
         -0.9996, -0.9909, -1.0000,  0.9876, -0.9999, -0.9902, -0.6138, -0.0317,
         -0.5201, -0.9980,  0.9972,  0.5767,  0.9971,  0.4269,  0.9998, -1.0000,
          0.9980, -0.9988, -1.0000,  1.0000, -0.1794, -0.9999, -0.4547,  1.0000,
          1.0000,  1.0000,  0.6808, -0.9810,  0.8330,  0.9166, -0.9905,  0.0571,
         -0.3544, -0.9997,  0.9984, -0.9582,  0.1701, -1.0000,  0.5173, -0.9916,
          0.9992,  0.1550,  0.6518, -0.8142,  1.0000, -0.6597, -0.6785,  0.9921,
          0.9964, -0.8139,  0.9997,  1.0000,  1.0000,  0.9253,  0.9995, -0.9999,
          0.9993,  0.5017,  0.9986, -0.9969,  0.9975,  1.0000,  0.9106, -1.0000,
         -0.9665, -0.9999,  0.5269, -0.8332, -0.7432,  0.1985,  1.0000,  0.9987,
         -0.9986, -0.9980, -

In [22]:
# (1,6) @ (6, 128) -> si le se suma 'algo de 128' se conoce como broadcasting rules
# la suma de una listas de listas y una lista, siempre y cuando:
# compartan las mismas dimensiones, o que una de las dimensiones sea 1 o una de las dimensiones no existe
# cada tensor tiene por lo menos una dimension


In [23]:
torch.tensor([[1,2,3,4], [2,3,4,5]]) + torch.tensor([1,2,3,4])
torch.tensor([[1,2,3,4], [2,3,4,5]]).shape, torch.tensor([1,2,3,4]).shape
# 2, 4(en este caso los 4 tenian que ser iguales)
# X, 4

(torch.Size([2, 4]), torch.Size([4]))

In [24]:
#capa output

W2 = torch.randn(n_hidden, charset_size)
b2 = torch.randn(charset_size)


In [25]:
act = F.tanh(emb[0].view(-1, 6) @ W1 + b1 )

In [26]:
#(13,128) @ (128, 40) --> (13, 40)
logits = act @ W2 + b2
counts = logits.exp()
probs = counts/counts.sum(axis=1, keepdims = True)

#e ** log(x)

In [27]:
probs

tensor([[1.1929e-07, 3.0519e-06, 3.3750e-13, 5.4118e-13, 4.2322e-09, 6.1422e-10,
         1.6610e-04, 1.0762e-03, 1.4699e-05, 8.0534e-09, 2.1378e-12, 2.1338e-11,
         7.4527e-19, 1.5441e-10, 1.3621e-10, 7.6524e-10, 3.6622e-12, 7.3039e-01,
         3.2799e-09, 5.5314e-15, 8.1959e-17, 9.8527e-15, 4.7787e-06, 5.3470e-10,
         1.6558e-06, 3.9379e-09, 9.6830e-17, 5.7275e-11, 2.7223e-02, 7.8011e-06,
         1.9505e-01, 8.8015e-06, 4.3832e-05, 2.9974e-07, 4.5557e-02, 4.4603e-04,
         9.2008e-11, 1.9117e-17, 3.3511e-07, 4.3429e-07]])

In [48]:
#pasando a limpio
context_size = 3 #tamaño del contexto o cuatos caracteres vamos a usar para predecir el siguiente
X, Y = [], []

for d in dataset[:1]:
    #print(d)
    example = list(d) + ['*']
    context = [0] * context_size
    for c in example:
        #print(''.join([itoc[x] for x in context])+ '---> + c')
        X.append(context)
        Y.append(ctoi[c])
        context = context[1:] + [ctoi[c]]
X = torch.tensor(X)
Y = torch.tensor(Y)

In [57]:
import torch.nn.functional as F

class Model:
    def __init__(self, charset_size, context_size, emb_size, hidden_size, g=torch.Generator().manual_seed(42)):
        self.charset_size = charset_size #tamaño de output/modelo
        self.context_size = context_size #cuantos caracteres mira atras
        self.emb_size = emb_size #representacion de cada caracter(vector de emb_size columnas)
        self.hidden_size = hidden_size #tamaño de la capa oculta
        self.C = torch.randn(self.charset_size, self.emb_size, generator=g)
        self.W1 = torch.randn(self.emb_size*self.context_size, self.hidden_size, generator=g)
        self.b1 = torch.randn(self.hidden_size, generator=g)
        self.W2 = torch.randn(self.hidden_size, self.charset_size, generator=g)
        self.b2 = torch.randn(1, self.charset_size, generator=g)
        self.parameters = [self.C, self.W1, self.b1, self.W2, self.b2] #calcular cantidad de parametros de modelo
        for p in self.parameters:
            p.requires_grad = True

    def count_parameters(self):
        return sum([p.nelement() for p in self.parameters])

    def __call__(self, X):
        if not torch.is_tensor(X):
            X = torch.tensor(X)
        if X.dim() == 1:
            X = X.unsqueeze(0)
        X = X.long()
        emb = self.C[X]
        emb = emb.view(X.shape[0], -1)
        act = torch.tanh(emb @ self.W1 + self.b1)
        logits = act @ self.W2 + self.b2
        return logits

def nll(logits, Y):
    return F.cross_entropy(logits, Y.long())


In [50]:
W1.nelement(), 6*128

(768, 768)

In [53]:
model = Model(charset_size,context_size, emb_size, n_hidden)
model(X[0])

tensor([[-11.6331, -10.0629,  11.3933,   6.5970,  14.6223,  -6.4226,  12.0905,
          -2.1454,   4.0184,   1.9550,   1.0228,   2.8076, -17.3637,  13.3137,
           2.2958, -13.5301, -10.5329,   3.1635,  -1.5997,   3.0007,  -4.5677,
          -3.9613,  -1.2003, -12.2594,  -1.1878,  -5.8063,  -5.4704,  21.4449,
           8.7539,   2.8296,   5.5351,   2.1804,   5.5357, -20.5130,  -6.8868,
           8.5803,   4.5198,  16.0104,   8.9889,   2.1782],
        [-11.3890,  -6.2400,  13.0468,   7.1670,  12.5245,  -5.5182,  10.9652,
          -2.0068,   5.4730,   2.6422,   2.5776,   4.8925, -18.3536,  13.6446,
           3.7459, -13.8270, -11.1442,   2.4405,   0.4753,   2.3110,  -4.0568,
          -5.1100,  -1.7465, -11.6216,  -3.2182,  -7.4176,  -4.5718,  22.6008,
           5.9945,  -0.5269,   7.5733,   2.6515,   5.5766, -20.8182,  -7.7094,
           8.4153,   3.0796,  14.2352,   8.4526,   2.8399],
        [-11.1610,  -7.3160,  14.3972,   8.2792,  13.6554,  -5.9970,  11.0342,
          -

In [54]:
model.count_parameters()

6216

In [55]:
logits = model(X[0])
logits

tensor([[-11.6331, -10.0629,  11.3933,   6.5970,  14.6223,  -6.4226,  12.0905,
          -2.1454,   4.0184,   1.9550,   1.0228,   2.8076, -17.3637,  13.3137,
           2.2958, -13.5301, -10.5329,   3.1635,  -1.5997,   3.0007,  -4.5677,
          -3.9613,  -1.2003, -12.2594,  -1.1878,  -5.8063,  -5.4704,  21.4449,
           8.7539,   2.8296,   5.5351,   2.1804,   5.5357, -20.5130,  -6.8868,
           8.5803,   4.5198,  16.0104,   8.9889,   2.1782],
        [-11.3890,  -6.2400,  13.0468,   7.1670,  12.5245,  -5.5182,  10.9652,
          -2.0068,   5.4730,   2.6422,   2.5776,   4.8925, -18.3536,  13.6446,
           3.7459, -13.8270, -11.1442,   2.4405,   0.4753,   2.3110,  -4.0568,
          -5.1100,  -1.7465, -11.6216,  -3.2182,  -7.4176,  -4.5718,  22.6008,
           5.9945,  -0.5269,   7.5733,   2.6515,   5.5766, -20.8182,  -7.7094,
           8.4153,   3.0796,  14.2352,   8.4526,   2.8399],
        [-11.1610,  -7.3160,  14.3972,   8.2792,  13.6554,  -5.9970,  11.0342,
          -

In [58]:
nll(model(X), Y)

RuntimeError: The size of tensor a (13) must match the size of tensor b (3) at non-singleton dimension 0

In [ ]:
#1. Forwards pass
logits = model(X)

#2. Loss
loss = nll(logits, Y)
loss

#3. zero grad